In [ ]:
import sys
import os
import networkx as nx
import html
import re
from typing import Any, Dict, List, Optional

sys.path.append("C:\github\cmf\doc-proc-solution-accelerator\doc-proc-lib")  # noqa: E402

In [ ]:
os.environ["APP_CONFIGURATION_URI"] = "https://appcs-REPLACE_ME.azconfig.io"  # noqa: E402

from dependencies import get_config
config = get_config()

In [ ]:
from connectors.postgres import PostgresClient
from sqlalchemy.orm import Session
from sqlalchemy import text
from collections.abc import Mapping

In [ ]:
settings = {
    "host": config.get("POSTGRES_HOST"),
    #"port": config.get("POSTGRES_PORT", 5432, int, True),
    "database": config.get("POSTGRES_DATABASE_NAME"),
    "username": config.get("POSTGRES_USERNAME"),
    "password": config.get("POSTGRES_PASSWORD"),
    "sslmode": config.get("POSTGRES_SSLMODE", "require"),
    "auth_type": config.get("POSTGRES_AUTH_TYPE", "password")
}

postgres_client = PostgresClient(**settings) 
postgres_client.connect()

In [ ]:
def clean_str(input: Any) -> str:
    """Clean an input string by removing HTML escapes, control characters, and other unwanted characters."""
    # If we get non-string input, just give it back
    if not isinstance(input, str):
        return input

    result = html.unescape(input.strip())
    # https://stackoverflow.com/questions/4324790/removing-control-characters-from-a-string-in-python
    return re.sub(r"[\x00-\x1f\x7f-\x9f]", "", result)

def _unpack_descriptions(data: Mapping) -> list[str]:
    value = data.get("description", None)
    return [] if value is None else value.split("\n")


def _unpack_source_ids(data: Mapping) -> list[str]:
    value = data.get("source_id", None)
    return [] if value is None else value.split(", ")

In [ ]:
#get all the records from the table
session = Session(postgres_client.engine)
sql = "SELECT id, parent_id, document_ids, entity_ids, relationship_ids FROM public.vectors;"
results = session.execute(text(sql))
records = results.fetchall()

In [ ]:
graph = nx.Graph()
join_descriptions = True

In [ ]:
for record in records:
    source_doc_id = record.parent_id
    document_ids = record.document_ids
    entity_ids = record.entity_ids
    relationship_ids = record.relationship_ids

    for entity_id in entity_ids:
        entity_name = clean_str(entity_id.get('title', ''))
        entity_type = clean_str(entity_id.get('type', ''))
        entity_description = clean_str(entity_id.get('description', ''))

        if entity_name in graph.nodes():
            node = graph.nodes[entity_name]
            if join_descriptions:
                node["description"] = "\n".join(
                    list({
                        *_unpack_descriptions(node),
                        entity_description,
                    })
                )
            else:
                if len(entity_description) > len(node["description"]):
                    node["description"] = entity_description
            node["source_id"] = ", ".join(
                list({
                    *_unpack_source_ids(node),
                    str(source_doc_id),
                })
            )
            node["type"] = (
                entity_type if entity_type != "" else node["type"]
            )
        else:
            graph.add_node(
                entity_name,
                type=entity_type,
                description=entity_description,
                source_id=str(source_doc_id),
            )

    for relationship_id in relationship_ids:
        source = clean_str(relationship_id.get('source', '').upper())
        target = clean_str(relationship_id.get('target', '').upper())
        edge_description = clean_str(relationship_id.get('description', ''))
        edge_source_id = clean_str(str(source_doc_id))
        try:
            weight = float(relationship_id.get('weight', 1.0))
        except ValueError:
            weight = 1.0

        if source not in graph.nodes():
            graph.add_node(
                source,
                type="",
                description="",
                source_id=edge_source_id,
            )
        if target not in graph.nodes():
            graph.add_node(
                target,
                type="",
                description="",
                source_id=edge_source_id,
            )
        if graph.has_edge(source, target):
            edge_data = graph.get_edge_data(source, target)
            if edge_data is not None:
                weight += edge_data["weight"]
                if join_descriptions:
                    edge_description = "\n".join(
                        list({
                            *_unpack_descriptions(edge_data),
                            edge_description,
                        })
                    )
                edge_source_id = ", ".join(
                    list({
                        *_unpack_source_ids(edge_data),
                        str(source_doc_id),
                    })
                )
        graph.add_edge(
            source,
            target,
            weight=weight,
            description=edge_description,
            source_id=edge_source_id,
        )

In [ ]:
print(graph)

In [ ]:
def create_entity(tx, name: str, type: str, description: str, source_id: str):
    tx.run(
        "MERGE (e:Entity {name: $name}) "
        "SET e.type = $type, e.description = $description, e.source_id = $source_id",
        name=name, type=type, description=description, source_id=source_id
    )

def create_relationship(tx, source: str, target: str, weight: float, description: str, source_id: str):
    tx.run(
        "MATCH (a:Entity {name: $source}), (b:Entity {name: $target}) "
        "MERGE (a)-[r:RELATED_TO]->(b) "
        "SET r.weight = $weight, r.description = $description, r.source_id = $source_id",
        source=source, target=target, weight=weight, description=description, source_id=source_id
    )

In [ ]:
#save to neo4j
from neo4j import GraphDatabase, basic_auth

uri = "bolt://localhost:7687"
user = "neo4j"
password = "Seattle123"

driver = GraphDatabase.driver(uri, auth=basic_auth(user, password))

with driver.session() as session:
    for node, data in graph.nodes(data=True):
        session.execute_write(
            create_entity,
            name=node,
            type=data.get("type", ""),
            description=data.get("description", ""),
            source_id=data.get("source_id", "")
        )
    for source, target, data in graph.edges(data=True):
        session.execute_write(
            create_relationship,
            source=source,
            target=target,
            weight=data.get("weight", 1.0),
            description=data.get("description", ""),
            source_id=data.get("source_id", "")
        )

In [ ]:
#create commnunities
from networkx.algorithms import community
communities = community.greedy_modularity_communities(graph)
for i, comm in enumerate(communities):
    print(f"Community {i+1}: {comm}")

In [ ]:
from graspologic.partition import hierarchical_leiden

from graphrag.index.utils.stable_lcc import stable_largest_connected_component

Communities = list[tuple[int, int, int, list[str]]]


In [ ]:
import logging
log = logging.getLogger(__name__)

In [ ]:
def cluster_graph(
    graph: nx.Graph,
    max_cluster_size: int,
    use_lcc: bool,
    seed: int | None = None,
) -> Communities:
    """Apply a hierarchical clustering algorithm to a graph."""
    if len(graph.nodes) == 0:
        log.warning("Graph has no nodes")
        return []

    node_id_to_community_map, parent_mapping = _compute_leiden_communities(
        graph=graph,
        max_cluster_size=max_cluster_size,
        use_lcc=use_lcc,
        seed=seed,
    )

    levels = sorted(node_id_to_community_map.keys())

    clusters: dict[int, dict[int, list[str]]] = {}
    for level in levels:
        result = {}
        clusters[level] = result
        for node_id, raw_community_id in node_id_to_community_map[level].items():
            community_id = raw_community_id
            if community_id not in result:
                result[community_id] = []
            result[community_id].append(node_id)

    results: Communities = []
    for level in clusters:
        for cluster_id, nodes in clusters[level].items():
            results.append((level, cluster_id, parent_mapping[cluster_id], nodes))
    return results


# Taken from graph_intelligence & adapted
def _compute_leiden_communities(
    graph: nx.Graph | nx.DiGraph,
    max_cluster_size: int,
    use_lcc: bool,
    seed: int | None = None,
) -> tuple[dict[int, dict[str, int]], dict[int, int]]:
    """Return Leiden root communities and their hierarchy mapping."""
    if use_lcc:
        graph = stable_largest_connected_component(graph)

    community_mapping = hierarchical_leiden(
        graph, max_cluster_size=max_cluster_size, random_seed=seed
    )
    results: dict[int, dict[str, int]] = {}
    hierarchy: dict[int, int] = {}
    for partition in community_mapping:
        results[partition.level] = results.get(partition.level, {})
        results[partition.level][partition.node] = partition.cluster

        hierarchy[partition.cluster] = (
            partition.parent_cluster if partition.parent_cluster is not None else -1
        )

    return results, hierarchy

In [ ]:
cluster_graph(
    graph=graph,
    max_cluster_size=10,
    use_lcc=True,
    seed=42,
)

In [ ]:
#save to neo4j
from connectors.neo4j import Neo4jClient

neo4j_client = Neo4jClient(
    uri=config.get("NEO4J_URI"),
    username=config.get("NEO4J_USERNAME"),
    password=config.get("NEO4J_PASSWORD"),
    auth_type=config.get("NEO4J_AUTH_TYPE", "basic")
)

neo4j_client.save_graph(graph)  #saves the graph to neo4j